<a href="https://colab.research.google.com/github/jiminmini/mini/blob/ESAA_OB/10_3_%ED%95%84%EC%82%AC_%EA%B3%BC%EC%A0%9C%20%EC%B5%9C%EC%A2%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**[개념 정리]**

##**[문서 군집화]**
- 비슷한 텍스트 구성의 문서를 군집화 하는 것

##**[opinion review 데이터를 이용]**
- dataframe으로 로드 > 문서를 TF-IDF 형태로 피처 벡터화
- lemnormalize() 함수 생성


##**[군집별 핵심 단어 추출하기]**
- KMeans 객체: 구성 단어 피처가 군집의 중심 기준으로 얼마나 가깝게 위치해 있는지 제공

#**[코드 필사]**

In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


# 필수 NLTK 데이터 다운로드 (필요시)
# nltk.download('punkt')
# nltk.download('punkt_tab')
# nltk.download('wordnet')


def LemNormalize(text):
  lemmatizer = WordNetLemmatizer()
  return [lemmatizer.lemmatize(token) for token in word_tokenize(text)]

In [ ]:
import glob, os
import warnings
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 700)

# 다음은 저자의 컴퓨터에서 압축 파일을 풀어놓은 디렉터리이니, 각자 디렉터리를 다시 설정합니다.
path = '/content/drive/MyDrive/ESAA/OpinosisDataset1.0/topics'

# path로 지정한 디렉터리 밑에 있는 모든 .data 파일들의 파일명을 리스트로 취합
all_files = glob.glob(os.path.join(path, "*.data"))

filename_list = []
opinion_text = []

# 개별 파일들의 파일명은 filename_list 리스트로 취합,
# 개별 파일들의 파일 내용은 DataFrame 로딩 후 다시 string으로 변환하여 opinion_text 리스트로 취합
for file_ in all_files:
    # 개별 파일을 읽어서 DataFrame으로 생성
    df = pd.read_table(file_, index_col=None, header=0, encoding='latin1')

    # 경로에서 파일명만 추출하고 확장자 제거
    filename = os.path.splitext(os.path.basename(file_))[0]

    # 파일명 리스트와 파일 내용 리스트에 파일명과 파일 내용을 추가
    filename_list.append(filename)
    opinion_text.append(df.to_string())

# 파일명 리스트와 파일 내용 리스트를 DataFrame으로 생성
document_df = pd.DataFrame({'filename': filename_list, 'opinion_text': opinion_text})
document_df.head()


Mounted at /content/drive


,filename,opinion_text
0,screen_ipod_nano_8gb.txt,"As always, the video screen is sharp and bright .\n0 2, inch screen and a glossy, polished aluminum finish that one CNET editor described as looking like a Christmas tree ornament .\n1 ..."
1,display_garmin_nuvi_255W_gps.txt,"3 quot widescreen display was a bonus .\n0 This made for smoother graphics on the 255w of the vehicle moving along displayed roads, where the 750's display was more of a jerky movement .\n1 ..."
2,gas_mileage_toyota_camry_2007.txt,Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 ...
3,free_bestwestern_hotel_sfo.txt,The wine reception is a great idea as it is nice to meet other travellers and great having access to the free Internet access in our room .\n0 They also have a computer available with free internet which is a nice bonus but I didn't find that out till the day before we left but was still able to get on there to check our flight to Vegas the next day .\n1 ...
4,food_swissotel_chicago.txt,The food for our event was delicious .\n0 ...


In [ ]:
import re
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk

# NLTK 패키지 다운로드
nltk.download('punkt')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def LemNormalize(text):
    # 소문자 변환
    text = text.lower()
    # 영어 알파벳과 숫자만 남기고 나머지는 공백으로
    text = re.sub(r"[^a-z0-9]", " ", text)
    # 토큰화
    tokens = word_tokenize(text)
    # 빈 토큰 제거
    tokens = [token for token in tokens if token.strip() != '']
    # 표제어 변환
    return [lemmatizer.lemmatize(token) for token in tokens]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

tfidf_vect = TfidfVectorizer(
    tokenizer=LemNormalize,
    stop_words=None,
    ngram_range=(1,2),
    min_df=1,
    max_df=0.95
)

feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])


In [ ]:
from sklearn.cluster import KMeans
 # 5개 집합으로 군집화 수행. 예제를 위해 동일한 클러스터링 결과 도출용 random_state게
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_
cluster_centers = km_cluster.cluster_centers_

In [ ]:
document_df['cluster_label'] = cluster_label
document_df.head()

,filename,opinion_text,cluster_label
0,screen_ipod_nano_8gb.txt,"As always, the video screen is sharp and bright .\n0 2, inch screen and a glossy, polished aluminum finish that one CNET editor described as looking like a Christmas tree ornament .\n1 ...",0
1,display_garmin_nuvi_255W_gps.txt,"3 quot widescreen display was a bonus .\n0 This made for smoother graphics on the 255w of the vehicle moving along displayed roads, where the 750's display was more of a jerky movement .\n1 ...",1
2,gas_mileage_toyota_camry_2007.txt,Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 ...,1
3,free_bestwestern_hotel_sfo.txt,The wine reception is a great idea as it is nice to meet other travellers and great having access to the free Internet access in our room .\n0 They also have a computer available with free internet which is a nice bonus but I didn't find that out till the day before we left but was still able to get on there to check our flight to Vegas the next day .\n1 ...,3
4,food_swissotel_chicago.txt,The food for our event was delicious .\n0 ...,4


In [ ]:
document_df [document_df ['cluster_label' ]==0] .sort_values(by='filename')

,filename,opinion_text,cluster_label
46,battery-life_amazon_kindle.txt,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever !\n0 After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there !\n1 ...",0
17,battery-life_ipod_nano_8gb.txt,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...,0
16,battery-life_netbook_1005ha.txt,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh Li, ion Battery , and a 1 .\n0 Not to mention that as of now...",0
21,features_windows7.txt,"I had to uninstall anti, virus and selected other programs, some of which did not have listings in the Programs and Features Control Panel section .\n0 This review briefly touches upon some of the key features and enhancements of Microsoft's latest OS .\n1 ...",0
32,keyboard_netbook_1005ha.txt,", I think the new keyboard rivals the great hp mini keyboards .\n0 Since the battery life difference is minimum, the only reason to upgrade would be to get the better keyboard .\n1 The keyboard is now as good as t...",0
35,performance_netbook_1005ha.txt,"The Eee Super Hybrid Engine utility lets users overclock or underclock their Eee PC's to boost performance or provide better battery life depending on their immediate requirements .\n0 In Super Performance mode CPU, Z shows the bus speed to increase up to 169 .\n1 One...",0
49,screen_garmin_nuvi_255W_gps.txt,It is easy to read and when touching the screen it works great !\n0 and zoom out buttons on the 255w to the same side of the screen which makes it a bit easier .\n1 ...,0
0,screen_ipod_nano_8gb.txt,"As always, the video screen is sharp and bright .\n0 2, inch screen and a glossy, polished aluminum finish that one CNET editor described as looking like a Christmas tree ornament .\n1 ...",0
27,screen_netbook_1005ha.txt,Keep in mind that once you get in a room full of light or step outdoors screen reflections could become annoying .\n0 I've used mine outsi...,0
8,size_asus_netbook_1005ha.txt,"A few other things I'd like to point out is that you must push the micro, sized right angle end of the ac adapter until it snaps in place or the battery may not charge .\n0 The full size right shift k...",0


In [ ]:
document_df [document_df [ 'cluster_label' ]==1 ] .sort_values(by='filename')

,filename,opinion_text,cluster_label
5,accuracy_garmin_nuvi_255W_gps.txt,", and is very, very accurate .\n0 but for the most part, we find that the Garmin software provides accurate directions, whereever we intend to go .\n1 This functi...",1
30,directions_garmin_nuvi_255W_gps.txt,You also get upscale features like spoken directions including street names and programmable POIs .\n0 I used to hesitate to go out of my directions but no...,1
1,display_garmin_nuvi_255W_gps.txt,"3 quot widescreen display was a bonus .\n0 This made for smoother graphics on the 255w of the vehicle moving along displayed roads, where the 750's display was more of a jerky movement .\n1 ...",1
2,gas_mileage_toyota_camry_2007.txt,Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 ...,1
50,mileage_honda_accord_2008.txt,"It's quiet, get good gas mileage and looks clean inside and out .\n0 The mileage is great, and I've had to get used to stopping less for gas .\n1 Thought gas ...",1
14,satellite_garmin_nuvi_255W_gps.txt,"It's fast to acquire satellites .\n0 If you've ever had a Brand X GPS take you on some strange route that adds 20 minutes to your trip, has you turn the wrong way down a one way road, tell you to turn AFTER you've passed the street, frequently loses the satellite signal, or has old maps missing streets, you know how important this stuff is .\n1 ...",1
37,speed_garmin_nuvi_255W_gps.txt,Another feature on the 255w is a display of the posted speed limit on the road which you are currently on right above your current displayed speed .\n0 I found myself not even looking at my car speedometer as I could easily see my current speed and the speed limit of my route at a glance .\n1 ...,1
22,updates_garmin_nuvi_255W_gps.txt,"Another thing to consider was that I paid $50 less for the 750 and it came with the FM transmitter cable and a USB cord to connect it to your computer for updates and downloads .\n0 update and reroute much _more_ quickly than my other GPS .\n1 UPDATE ON THIS , It finally turned out that to see the elevation contours at lowe...",1
28,voice_garmin_nuvi_255W_gps.txt,The voice prompts and maps are wonderful especially when driving after dark .\n0 I also thought the the voice prompts of the 750 where more pleasant sounding than the 255w's .\n1 ...,1


In [ ]:
document_df [document_df[ 'cluster_label']==2].sort_values (by='filename')

,filename,opinion_text,cluster_label
13,buttons_amazon_kindle.txt,"I thought it would be fitting to christen my Kindle with the Stephen King novella UR, so went to the Amazon site on my computer and clicked on the button to buy it .\n0 As soon as I'd clicked the button to confirm my order it appeared on my Kindle almost immediately !\n1 ...",2
41,eyesight-issues_amazon_kindle.txt,"It feels as easy to read as the K1 but doesn't seem any crisper to my eyes .\n0 the white is really GREY, and to avoid considerable eye, strain I had to refresh pages every other page .\n1 The dream has always been a portable electronic device that could hold a ton of reading material, automate subscriptions and fa...",2
33,fonts_amazon_kindle.txt,"Being able to change the font sizes is awesome !\n0 For whatever reason, Amazon decided to make the Font on the Home Screen ...",2
12,navigation_amazon_kindle.txt,"In fact, the entire navigation structure has been completely revised , I'm still getting used to it but it's a huge step forward .\n0 ...",2
11,performance_honda_accord_2008.txt,"Very happy with my 08 Accord, performance is quite adequate it has nice looks and is a great long, distance cruiser .\n0 6, 4, 3 eco engine has poor performance and gas mileage of 22 highway .\n1 Overall performance is good but comfort level is poor .\n2 ...",2
20,price_amazon_kindle.txt,"If a case was included, as with the Kindle 1, that would have been reflected in a higher price .\n0 lower overall price, with nice leather cover .\n1 ...",2
43,price_holiday_inn_london.txt,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price .\n0 ...",2
19,quality_toyota_camry_2007.txt,I previously owned a Toyota 4Runner which had incredible build quality and reliability .\n0 I bought the Camry because of Toyota reliability and qua...,2


In [ ]:
document_df [document_df[ 'cluster_label']==3].sort_values (by='filename')

,filename,opinion_text,cluster_label
45,food_holiday_inn_london.txt,The room was packed to capacity with queues at the food buffets .\n0 The over zealous staff cleared our unfinished drinks while we were collecting cooked food and movement around the room with plates was difficult in the crowded circumstances .\n1 ...,3
3,free_bestwestern_hotel_sfo.txt,The wine reception is a great idea as it is nice to meet other travellers and great having access to the free Internet access in our room .\n0 They also have a computer available with free internet which is a nice bonus but I didn't find that out till the day before we left but was still able to get on there to check our flight to Vegas the next day .\n1 ...,3
42,location_bestwestern_hotel_sfo.txt,"Good Value good location , ideal choice .\n0 Great Location , Nice Rooms , Helpless Concierge\n1 ...",3
47,service_bestwestern_hotel_sfo.txt,"Both of us having worked in tourism for over 14 years were very disappointed at the level of service provided by this gentleman .\n0 The service was good, very friendly staff and we loved the free wine reception each night .\n1 ...",3
18,staff_bestwestern_hotel_sfo.txt,Staff are friendl...,3
48,staff_swissotel_chicago.txt,"The staff at Swissotel were not particularly nice .\n0 Each time I waited at the counter for staff for several minutes and then was waved to the desk upon my turn with no hello or anything, or apology for waiting in line .\n1 ...",3


In [ ]:
document_df [document_df[ 'cluster_label']==4].sort_values (by='filename')

,filename,opinion_text,cluster_label
44,bathroom_bestwestern_hotel_sfo.txt,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ...",4
34,comfort_honda_accord_2008.txt,"Drivers seat not comfortable, the car itself compared to other models of similar class .\n0 ...",4
6,comfort_toyota_camry_2007.txt,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 Seats are fine, in fact of all the smaller sedans this is the most comfortable I found for the price as I am 6', 2 and 250# .\n1 Great gas mileage and comfortable on long trips ...",4
4,food_swissotel_chicago.txt,The food for our event was delicious .\n0 ...,4
9,interior_honda_accord_2008.txt,I love the new body style and the interior is a simple pleasure except for the center dash .\n0 ...,4
10,interior_toyota_camry_2007.txt,"First of all, the interior has way too many cheap plastic parts like the cheap plastic center piece that houses the clock .\n0 3 blown struts at 30,000 miles, interior trim coming loose and rattling squeaking, stains on paint, and bug splats taking paint off, premature uneven brake wear, on 3rd windsh...",4
26,location_holiday_inn_london.txt,"Great location for tube and we crammed in a fair amount of sightseeing in a short time .\n0 All in all, a normal chain hotel on a nice lo...",4
39,parking_bestwestern_hotel_sfo.txt,Parking was expensive but I think this is common for San Fran .\n0 there is a fee for parking but well worth it seeing no where to park if you do have a car .\n1 ...,4
7,room_holiday_inn_london.txt,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave u...",4
38,rooms_bestwestern_hotel_sfo.txt,"Great Location , Nice Rooms , H...",4


In [ ]:
from sklearn.cluster import KMeans

 # 3개의 집합으로 군집화
km_cluster = KMeans(n_clusters=3, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_

 # 소속 클러스터를 cl니ster_label 칼럼으로 할당하고 cluster_label 값으로 정렬
document_df ['cluster_label'] = cluster_label
document_df.sort_values(by='cluster_label')

,filename,opinion_text,cluster_label
0,screen_ipod_nano_8gb.txt,"As always, the video screen is sharp and bright .\n0 2, inch screen and a glossy, polished aluminum finish that one CNET editor described as looking like a Christmas tree ornament .\n1 ...",0
10,interior_toyota_camry_2007.txt,"First of all, the interior has way too many cheap plastic parts like the cheap plastic center piece that houses the clock .\n0 3 blown struts at 30,000 miles, interior trim coming loose and rattling squeaking, stains on paint, and bug splats taking paint off, premature uneven brake wear, on 3rd windsh...",0
9,interior_honda_accord_2008.txt,I love the new body style and the interior is a simple pleasure except for the center dash .\n0 ...,0
8,size_asus_netbook_1005ha.txt,"A few other things I'd like to point out is that you must push the micro, sized right angle end of the ac adapter until it snaps in place or the battery may not charge .\n0 The full size right shift k...",0
24,speed_windows7.txt,"Windows 7 is quite simply faster, more stable, boots faster, goes to sleep faster, comes back from sleep faster, manages your files better and on top of that it's beautiful to look at and easy to use .\n0 , faster about 20% to 30% faster at running applications than my Vista , seriously\n1 ...",0
21,features_windows7.txt,"I had to uninstall anti, virus and selected other programs, some of which did not have listings in the Programs and Features Control Panel section .\n0 This review briefly touches upon some of the key features and enhancements of Microsoft's latest OS .\n1 ...",0
17,battery-life_ipod_nano_8gb.txt,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...,0
16,battery-life_netbook_1005ha.txt,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh Li, ion Battery , and a 1 .\n0 Not to mention that as of now...",0
31,transmission_toyota_camry_2007.txt,"After slowing down, transmission has to be kicked to speed up .\n0 ...",0
27,screen_netbook_1005ha.txt,Keep in mind that once you get in a room full of light or step outdoors screen reflections could become annoying .\n0 I've used mine outsi...,0


In [ ]:
cluster_centers = km_cluster.cluster_centers_
print('cluster_centers shape ：', cluster_centers.shape)
print(cluster_centers)

cluster_centers shape ： (3, 63774)
[[0.00138517 0.0006969  0.00022494 ... 0.00280085 0.00154758 0.00138517]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


In [ ]:
# 군집별 top n 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명을 반환함.
def get_cluster_details(cluster_model, cluster_data, feature_names,
                        clusters_num, top_n_features=10):
    cluster_details = {}
 # cluster.centers array의 값이 큰 순으로 정렬된 인덱스 값을 반환
# 군집 중심점(centroid)별 할당된 word 피처들의 거리값이 큰 순으로 값을 구하기 위함.
    centroid_feature_ordered_ind = cluster_model.cluster_centers_.argsort()[:,:-1]
 # 개별 군집별로 반복하면서 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명 입력
    for cluster_num in range(clusters_num):
# 개별 군집별 정보를 담을 데이터 초기화.
        cluster_details[cluster_num] = {}
        cluster_details[cluster_num]['cluster'] = cluster_num
 # cluster_centers_.argsort()[：z ：：-1]로 구한 인덱스를 이용해 top n 피처 단어를 구함.
        top_feature_indexes = centroid_feature_ordered_ind[cluster_num, :top_n_features]
        top_features = [ feature_names[ind] for ind in top_feature_indexes ]
 # top_feature_indexes를 이용해 해당 피처 단어의 중심 위치 상댓값 구함.
        top_feature_values = cluster_model.cluster_centers_[cluster_num,
                                                        top_feature_indexes].tolist()
 # cluster_details 딕셔너리 객체에 개별 군집별 핵심단어와 중심위치 상댓값, 해당 파일명 입력
        cluster_details[cluster_num] [ 'top_features' ] = top_features
        cluster_details[cluster_num]['top_features_value'] = top_feature_values
        filenames = cluster_data[cluster_data[ 'cluster_label' ] == cluster_num] [ 'filename' ]
        filenames = filenames.values.tolist()

        cluster_details[cluster_num]['filenames'] = filenames
    return cluster_details

In [ ]:
def print_cluster_details(cluster_details):
    for cluster_num, cluster_detail in cluster_details.items():
        print('####### Cluster {0}',format(cluster_num))
        print('Top features：', cluster_detail['top_features' ])
        print('Reviews 파일명 ：', cluster_detail['filenames'][:7])
        print('================================================')

In [ ]:
feature_names = tfidf_vect.get_feature_names_out()
cluster_details = get_cluster_details(cluster_model=km_cluster,
                                      cluster_data=document_df, feature_names=feature_names, clusters_num=3, top_n_features=10 )
print_cluster_details(cluster_details)

####### Cluster {0} 0
Top features： ['hotel 483', 'hotel 369', 'hotel 376', 'hotel 377', 'hotel 386', 'hotel 388', 'hotel 4', 'hotel 42', 'hotel 442', 'hotel 45']
Reviews 파일명 ： ['screen_ipod_nano_8gb.txt', 'size_asus_netbook_1005ha.txt', 'interior_honda_accord_2008.txt', 'interior_toyota_camry_2007.txt', 'battery-life_netbook_1005ha.txt', 'battery-life_ipod_nano_8gb.txt', 'features_windows7.txt']
####### Cluster {0} 1
Top features： ['immaculately maintained', 'started clicking', 'started crashing', 'started dropping', 'immediately but', 'immediately called', 'immediately hit', 'immediately light', 'immediately remedied', 'immediately starting']
Reviews 파일명 ： ['display_garmin_nuvi_255W_gps.txt', 'gas_mileage_toyota_camry_2007.txt', 'accuracy_garmin_nuvi_255W_gps.txt', 'comfort_toyota_camry_2007.txt', 'satellite_garmin_nuvi_255W_gps.txt', 'seats_honda_accord_2008.txt', 'staff_bestwestern_hotel_sfo.txt']
####### Cluster {0} 2
Top features： ['many recharges', 'many interior', 'many languag